# Sandbox Program O-R — experimentos de David (DMPLA / SAC / lo que quieras)

**Qué es esto:** el MISMO entorno, observación y métricas del contrato congelado
`program_o_ret_only_learner_v1`, pero en un **sandbox de desarrollo**: seeds no científicas,
nada de lo que corras aquí es promovible ni contamina el experimento real.

**Reglas (automáticas, no negociables):**
1. Solo seeds de desarrollo `9491xxxxx`/`9492xxxxx` — el notebook REHÚSA tocar las series
   científicas (`748*` entrenamiento congelado, `7480001-48` calibración, `7480101-48` virgin,
   `747*` atlas). La celda de guardia lo verifica.
2. Un resultado bueno aquí = **hipótesis** para proponer un contrato preregistrado con seeds
   frescas y el evaluador real (enmienda v1_1). Nunca un claim.
3. La observación es la del contrato: SIN régimen latente, SIN demanda futura, SIN seed/celda.

**Cómo meter TU modelo:** edita solo la celda marcada `👉 TU AGENTE AQUÍ`.
- SB3 (PPO/A2C/RecurrentPPO/...) con tu arquitectura vía `policy_kwargs`.
- SAC: ojo — la acción es **discreta** (k ∈ {0,1,2,3}); SAC vanilla es continuo. Usa una
  implementación SAC-Discrete o tu propia clase PyTorch con la plantilla `MiAgente`.
- PyTorch puro: implementa `predict(obs) -> int` y opcionalmente `learn(env, steps)`.


In [ ]:
# ── Celda 1: setup + GUARDIA DE GOBERNANZA ────────────────────────────────────
import sys, json
from pathlib import Path
import numpy as np

ROOT = Path.cwd().parent if (Path.cwd().name == "notebooks") else Path.cwd()
sys.path.insert(0, str(ROOT))

# Seeds de desarrollo (fuera de TODA serie científica)
DEV_TRAIN_SEEDS = (949100001, 949199999)   # ~99k tapes: una fresca por episodio, como el contrato
DEV_EVAL_SEEDS  = list(range(949200001, 949200001 + 12))  # 12 tapes de evaluación

FORBIDDEN = [(747000000, 748999999), (7420001, 7430999), (7480001, 7480999)]
def assert_dev_seed(s):
    for lo, hi in FORBIDDEN:
        assert not (lo <= s <= hi), f"SEED PROHIBIDA {s}: rango científico [{lo},{hi}]"
for s in [DEV_TRAIN_SEEDS[0], DEV_TRAIN_SEEDS[1], *DEV_EVAL_SEEDS]:
    assert_dev_seed(s)
print("Guardia OK — solo seeds de desarrollo.")


In [ ]:
# ── Celda 2: entorno (idéntico al del contrato) ───────────────────────────────
from supply_chain.program_o_ret_env import ProgramORetOnlyEnv, CONFIRMED_RET_CELLS
from scripts.evaluate_program_o_ret_learner import (
    scheduler, trajectory_audit, derive_placebo_calendars, encode_calendar,
    GUARDRAIL_KEYS, RESOURCE_EQUALITY_KEYS,
)
from supply_chain.program_o_full_des_transducer import (
    extract_full_des_skeleton, full_action_calendars, simulate_full_des_frontier,
)

SCHED = scheduler()
def make_env():
    return ProgramORetOnlyEnv(
        scheduler=SCHED,
        tape_seed_start=DEV_TRAIN_SEEDS[0],
        tape_seed_end=DEV_TRAIN_SEEDS[1],
    )
env = make_env()
obs, info = env.reset()
print("obs shape:", obs.shape, "| acciones:", env.action_space, "| celdas:", [c.cell_id for c in CONFIRMED_RET_CELLS])


In [ ]:
# ── Celda 3: 👉 TU AGENTE AQUÍ (la única celda que necesitas editar) ─────────
# OPCIÓN A (default, funciona ya): SB3 PPO con arquitectura editable
from stable_baselines3 import PPO

TOTAL_TIMESTEPS = 20_000   # súbelo cuando tu agente prometa (200k = presupuesto del contrato)

agente = PPO(
    "MlpPolicy", make_env(), verbose=0, seed=8201,
    policy_kwargs=dict(net_arch=[64, 64]),   # 👈 tu arquitectura DMPLA aquí
)

# OPCIÓN B (PyTorch puro / SAC-Discrete / DMPLA from scratch): descomenta y edita.
# class MiAgente:
#     '''Contrato mínimo: predict(obs)->int en {0,1,2,3}; learn(env, steps) opcional.'''
#     def learn(self, env, total_timesteps):
#         obs, _ = env.reset()
#         for _ in range(total_timesteps):
#             action = int(np.random.randint(4))          # 👈 tu update aquí
#             obs, reward, done, trunc, info = env.step(action)
#             if done or trunc: obs, _ = env.reset()
#         return self
#     def predict(self, obs, deterministic=True):
#         return int(np.random.randint(4)), None           # 👈 tu forward aquí
# agente = MiAgente()


In [ ]:
# ── Celda 4: entrenar ─────────────────────────────────────────────────────────
agente.learn(TOTAL_TIMESTEPS) if hasattr(agente, "policy") else agente.learn(make_env(), TOTAL_TIMESTEPS)
print("entrenamiento listo:", TOTAL_TIMESTEPS, "timesteps")


In [ ]:
# ── Celda 5: evaluación con las métricas del contrato (tapes de eval NUEVAS) ──
from supply_chain.program_o_state_rich import finite_state_rich_configurations, state_rich_calendar

CELL = CONFIRMED_RET_CELLS[-1]              # rho90_share90 (la celda primaria)
ALL_CALS = full_action_calendars()
CONFIGS = finite_state_rich_configurations()

def rollout_calendar(agente, skeleton, cell_index):
    e = make_env()
    obs, _ = e.reset(options={"tape_seed": DEV_EVAL_SEEDS[0], "cell_index": cell_index,
                              "skeleton": skeleton})
    done, actions = False, []
    while not done:
        a = agente.predict(obs, deterministic=True)
        a = int(a[0]) if isinstance(a, tuple) else int(a)
        obs, r, done, trunc, info = e.step(a)
        actions.append(a)
    return tuple(actions)

rows, cals = [], []
cell_index = list(CONFIRMED_RET_CELLS).index(CELL)
for ts in DEV_EVAL_SEEDS:
    assert_dev_seed(ts)
    skeleton, _ = extract_full_des_skeleton(
        seed=ts, scheduler=SCHED, regime_persistence=CELL.regime_persistence,
        dominant_share=CELL.dominant_share,
        downstream_freight_physics_mode="fixed_clock_physical_v1")
    panel = simulate_full_des_frontier(skeleton=skeleton, scheduler=SCHED, calendars=ALL_CALS)
    cal = rollout_calendar(agente, skeleton, cell_index)
    cals.append(cal)
    me = {k: float(panel[k][encode_calendar(cal)]) for k in panel}
    best_cls = None
    for cfg in CONFIGS:
        c2, _ = state_rich_calendar(skeleton=skeleton.as_dict(), scheduler=SCHED, config=cfg,
                                    regime_persistence=0.75, dominant_share=0.90)
        v = float(panel["ret_visible"][encode_calendar(tuple(c2))])
        best_cls = max(best_cls or -1e9, v)
    res_spread = max(float(panel[k].max() - panel[k].min()) for k in RESOURCE_EQUALITY_KEYS)
    rows.append({"tape": ts, "yo": me, "open_loop_all": panel["ret_visible"],
                 "best_classical": best_cls, "resource_spread": res_spread})
print(f"evaluado en {len(rows)} tapes")


In [ ]:
# ── Celda 6: MÉTRICAS IMPORTANTES (lo único que hay que mirar) ────────────────
mi_ret       = np.array([r["yo"]["ret_visible"] for r in rows])
open_means   = np.stack([r["open_loop_all"] for r in rows]).mean(axis=0)
best_open_ix = int(open_means.argmax())
best_open    = np.array([r["open_loop_all"][best_open_ix] for r in rows])
best_cls     = np.array([r["best_classical"] for r in rows])

H_learned = float((mi_ret - best_open).mean())      # vs MEJOR calendario fijo (65,536)
H_neural  = float((mi_ret - best_cls).mean())       # vs mejor controlador clásico
audit     = trajectory_audit(cals)
plc       = derive_placebo_calendars(cals, rng_seed=8201)
plc_means = {k: float(np.stack([r["open_loop_all"] for r in rows])[:, encode_calendar(v)].mean())
             for k, v in plc.items()}
guard     = {k: float(np.mean([r["yo"][k] for r in rows])) for k in GUARDRAIL_KEYS}
res_dev   = max(r["resource_spread"] for r in rows)  # spread del frontier por tape: 0.0 = ninguna politica puede comprar recursos

print("═"*62)
print(f"  H_learned (vs mejor open-loop de 65,536): {H_learned:+.4f}   {'✅' if H_learned>0 else '❌'}")
print(f"  H_neural  (vs mejor clásico belief-MPC) : {H_neural:+.4f}   {'✅' if H_neural>=0 else '❌'}")
print(f"  tapes favorables vs open-loop           : {int((mi_ret>best_open).sum())}/{len(rows)}  (contrato pide ≥34/48 ≈ 71%)")
print(f"  ¿ADAPTA o descubrió un horario fijo?    : {audit['unique_calendars']} calendarios únicos, "
      f"modal {audit['modal_fraction']:.0%}  {'✅ adapta' if audit['passed'] else '❌ horario fijo'}")
print(f"  vs placebos: " + "  ".join(
      f"{k}={'✅' if mi_ret.mean()>v else '❌'}({v:.3f})" for k, v in plc_means.items()))
print(f"  guardrails (medias): " + "  ".join(f"{k}={v:.3f}" for k, v in guard.items()))
print(f"  recursos programados idénticos entre políticas: {'✅' if res_dev==0 else f'⚠️ dev={res_dev:.2e}'}")
print(f"  mi ReT medio: {mi_ret.mean():.4f}   | mejor open-loop: {best_open.mean():.4f}   | mejor clásico: {best_cls.mean():.4f}")
print("═"*62)
print("⚠️  SANDBOX: nada de esto es promovible. Si H_learned>0 y H_neural≥0 y '✅ adapta'")
print("    de forma estable → proponlo y se preregistra un contrato real con seeds frescas.")


## ¿Y ahora qué?

- **Itera en la Celda 3** (arquitectura, algoritmo, hiperparámetros). Todo lo demás es fijo.
- **La barra real** (contrato congelado + enmienda v1_1): vencer al frente completo de 65,536
  calendarios Y a los 10 controladores clásicos en 3 celdas, ≥34/48 tapes, ≥8/10 seeds del
  optimizador, placebos ejecutados y vencidos, audit de trayectoria, recursos exactamente
  iguales, en tapes selladas que nadie ha visto. Este notebook te da las mismas métricas en
  miniatura para iterar rápido.
- **Si tu DMPLA/SAC gana aquí de forma estable**: se propone al equipo, se preregistra un
  contrato con tu agente CONGELADO (arquitectura + hiperparámetros + seeds del optimizador),
  y se evalúa una sola vez con el evaluador real. Así tu resultado sería publicable y blindado.
